# 🎓 University Timetable Scheduling — CSP Backtracking Search

**Course:** Introduction to Artificial Intelligence  
**Case:** Case 1 — University Timetable Scheduling  
**Algorithm:** Constraint Satisfaction Problem (CSP) with Backtracking Search

---

## Problem Overview

Automatically generate a **conflict-free, optimised** weekly timetable for a university department.

### Resources
| Resource | Detail |
|---|---|
| Working Days | Sunday – Thursday (5 days) |
| Time Slots | 6 usable slots/day (Break between Slot 3 and Slot 4) |
| Rooms | 27 total (5 Lecture Halls, 6 Labs, 16 Classrooms) |

### Daily Slot Map
| Slot | Time | Status |
|---|---|---|
| Slot 1 | 09:00 – 09:50 | Teaching |
| Slot 2 | 09:50 – 10:40 | Teaching |
| Slot 3 | 10:40 – 11:30 | Teaching |
| **BREAK** | **11:30 – 12:10** | **Reserved** |
| Slot 4 | 12:10 – 13:00 | Teaching |
| Slot 5 | 13:00 – 13:50 | Teaching |
| Slot 6 | 13:50 – 14:40 | Teaching |

### Constraints
- **C1 (Room Conflict):** No two sessions in the same room at the same time.
- **C5 (Break Enforcement):** No session may cross the break (Slot 3 → Slot 4).
- **C6 (Instructor Conflict):** An instructor cannot teach two sessions at the same time.
- **S1 (Soft — Day Off):** Each instructor should have at least one full day off per week.

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import random
import copy
import time
from collections import defaultdict

print("✅ Libraries loaded.")

✅ Libraries loaded.


## Step 2 — Define University Resources & Slot Configuration

In [ ]:

# Working days
DAYS = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday"]

# 6 usable teaching slots per day (slots are 1-indexed for human readability)
# The break falls BETWEEN Slot 3 and Slot 4 — no session may straddle it.
SLOTS = [1, 2, 3, 4, 5, 6]

# Human-readable time labels for each slot
SLOT_TIMES = {
    1: "09:00-09:50",
    2: "09:50-10:40",
    3: "10:40-11:30",
    # --- BREAK: 11:30-12:10 ---
    4: "12:10-13:00",
    5: "13:00-13:50",
    6: "13:50-14:40",
}

# ─────────────────────────────────────────────────────────────────────────────
# BREAK ENFORCEMENT (Constraint C5)
# The break lives between Slot 3 and Slot 4.
# A session starting at slot S and occupying N slots uses slots S, S+1, ..., S+N-1.
# It crosses the break if it contains BOTH slot 3 AND slot 4.
# Valid contiguous blocks that respect the break:
#   Before break  → start slots: 1, 2, 3   (must finish by end of slot 3)
#   After  break  → start slots: 4, 5, 6   (must finish by end of slot 6)
#
# For a 2-slot class: valid starts = 1,2,3 (finishes ≤3) OR 4,5 (finishes ≤6)
#   → start 3 uses [3,4]  → CROSSES break → INVALID
# For a 3-slot class: valid starts = 1,2 (finishes ≤3) OR 4 (finishes ≤6)
#   → start 2 uses [2,3,4] → CROSSES break → INVALID
# ─────────────────────────────────────────────────────────────────────────────
BREAK_AFTER_SLOT = 3   # break falls after slot 3; slot 4 is post-break

# ─────────────────────────────────────────────────────────────────────────────
# ROOM INVENTORY
# Naming convention: LH = Lecture Hall, LAB = Laboratory, CR = Classroom
# ─────────────────────────────────────────────────────────────────────────────
LECTURE_HALLS = [f"LH-{i}"  for i in range(1, 6)]   # 5 lecture halls
LABS          = [f"LAB-{i}" for i in range(1, 7)]   # 6 labs
CLASSROOMS    = [f"CR-{i}"  for i in range(1, 17)]  # 16 classrooms
ALL_ROOMS     = LECTURE_HALLS + LABS + CLASSROOMS    # 27 rooms total

# Map session type → preferred room pool
# Sessions will be tried in their preferred pool first, then the full list.
ROOM_PREFERENCE = {
    "Lecture": LECTURE_HALLS + CLASSROOMS,   # lectures prefer halls, then classrooms
    "Lab":     LABS ,            # labs prefer lab rooms
    "Section": CLASSROOMS ,   # sections prefer classrooms
}

# ─────────────────────────────────────────────────────────────────────────────
# DURATION → SLOTS MAPPING
# Durations in the Excel file are stored as timedelta values.
# We map each duration to the number of consecutive slots it requires.
# ─────────────────────────────────────────────────────────────────────────────
DURATION_TO_SLOTS = {
    "0:50":  1,   # 50  minutes → 1 slot
    "1:40":  2,   # 100 minutes → 2 consecutive slots
    "2:30":  3,   # 150 minutes → 3 consecutive slots
}

print(f"✅ Resources defined:")
print(f"   Days        : {DAYS}")
print(f"   Slots/day   : {SLOTS}")
print(f"   Lecture Halls: {len(LECTURE_HALLS)}")
print(f"   Labs         : {len(LABS)}")
print(f"   Classrooms   : {len(CLASSROOMS)}")
print(f"   Total rooms  : {len(ALL_ROOMS)}")

✅ Resources defined:
   Days        : ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday']
   Slots/day   : [1, 2, 3, 4, 5, 6]
   Lecture Halls: 5
   Labs         : 6
   Classrooms   : 16
   Total rooms  : 27


## Step 3 — Load & Prepare Input Data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# The Excel file has columns: Total Time, Course Code, Course Name,
# Instructor, Session Type
# ─────────────────────────────────────────────────────────────────────────────
INPUT_FILE = "/content/ERU_Full_Schedule_Spring2026(1).xlsx"

df_raw = pd.read_excel(INPUT_FILE)

# Rename columns for clarity and consistency
df_raw.columns = ["Duration", "Course Code", "Course Name", "Instructor", "Session Type"]

# Standardize instructor names to fix capitalization and trailing space issues
df_raw["Instructor"] = df_raw["Instructor"].str.title().str.strip()
df_raw["Instructor"] = df_raw["Instructor"].str.replace("Rowaida", "Rowida")

# ─────────────────────────────────────────────────────────────────────────────
# DURATION CONVERSION
# The Duration column comes in as pandas timedelta. Convert to H:MM string
# that matches our DURATION_TO_SLOTS keys.
# ─────────────────────────────────────────────────────────────────────────────
def timedelta_to_key(td):
    """Convert a timedelta (e.g. 0 days 01:40:00) to a key like '1:40'."""
    total_minutes = int(td.total_seconds() // 60)
    hours   = total_minutes // 60
    minutes = total_minutes % 60
    return f"{hours}:{minutes:02d}"

df_raw["Duration Key"] = df_raw["Duration"].apply(timedelta_to_key)

# Map duration key → number of slots required
df_raw["Slots Required"] = df_raw["Duration Key"].map(DURATION_TO_SLOTS)

# ─────────────────────────────────────────────────────────────────────────────
# GENERATE UNIQUE SESSION ID
# The same course code can appear multiple times (lecture + lab + sections).
# We give every row its own unique ID: SESSION_0001, SESSION_0002, …
# ─────────────────────────────────────────────────────────────────────────────
df_raw.insert(0, "Session ID", [f"SES_{i+1:04d}" for i in range(len(df_raw))])

# Drop any rows where duration couldn't be mapped (safety check)
df = df_raw.dropna(subset=["Slots Required"]).copy()
df["Slots Required"] = df["Slots Required"].astype(int)

print(f"✅ Data loaded: {len(df)} sessions to schedule.")
print(f"\nDuration breakdown:")
print(df.groupby("Duration Key")[["Slots Required"]].agg(count=("Slots Required", "size")).reset_index())
print(f"\nSession type breakdown:")
print(df["Session Type"].value_counts().to_string())
print(f"\nUnique instructors: {df['Instructor'].nunique()}")
print("\nSample rows:")
df[["Session ID", "Course Code", "Instructor", "Session Type", "Duration Key", "Slots Required"]].head(8)

✅ Data loaded: 160 sessions to schedule.

Duration breakdown:
  Duration Key  count
0         0:50     37
1         1:40     84
2         2:30     39

Session type breakdown:
Session Type
Lecture    94
Section    37
Lab        29

Unique instructors: 66

Sample rows:


,Session ID,Course Code,Instructor,Session Type,Duration Key,Slots Required
0,SES_0001,IST101,Dr. Reham,Lecture,1:40,2
1,SES_0002,MIS408,T.A Paula,Lab,1:40,2
2,SES_0003,ECO204,Abdelkader,Lecture,1:40,2
3,SES_0004,ACC421,Nouf,Section,0:50,1
4,SES_0005,HM002,Dr. Halema,Lecture,1:40,2
5,SES_0006,DAS410,Dr. Yasser,Lecture,1:40,2
6,SES_0007,ACC308,Amr,Section,0:50,1
7,SES_0008,ARI301,Dr. Shady,Lecture,1:40,2


## Step 4 — CSP Helper Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VALID START-SLOT CALCULATION (Constraint C5 — Break Enforcement)
# ─────────────────────────────────────────────────────────────────────────────
def get_valid_start_slots(slots_required: int) -> list[int]:
    """
    Return the list of start slots where a session of `slots_required` length
    can begin WITHOUT crossing the mandatory break (C5).

    Break rule: No session may span both slot 3 (pre-break) and slot 4 (post-break).
    A session starting at `s` occupies slots [s, s+1, ..., s+slots_required-1].
    It violates C5 if it contains slot 3 AND slot 4 simultaneously.

    Valid windows:
      PRE-BREAK  window: slots 1..3  → session must END at or before slot 3
                         → start + slots_required - 1 <= 3
                         → start <= 4 - slots_required
      POST-BREAK window: slots 4..6  → session must START at or after slot 4
                         → start >= 4  AND start + slots_required - 1 <= 6
                         → start <= 7 - slots_required
    """
    valid = []

    # --- PRE-BREAK window ---
    for s in range(1, BREAK_AFTER_SLOT + 1):         # starts 1, 2, 3
        end_slot = s + slots_required - 1
        if end_slot <= BREAK_AFTER_SLOT:              # must finish BY slot 3
            valid.append(s)

    # --- POST-BREAK window ---
    post_break_start = BREAK_AFTER_SLOT + 1           # slot 4
    for s in range(post_break_start, len(SLOTS) + 1): # starts 4, 5, 6
        end_slot = s + slots_required - 1
        if end_slot <= len(SLOTS):                    # must finish BY slot 6
            valid.append(s)

    return valid


# Pre-compute valid start slots for each session length
VALID_STARTS = {n: get_valid_start_slots(n) for n in [1, 2, 3]}

print("Valid start slots per session length (respecting break after Slot 3):")
for n, starts in VALID_STARTS.items():
    occupied = [list(range(s, s+n)) for s in starts]
    print(f"  {n}-slot session → start slots {starts} → occupies {occupied}")

Valid start slots per session length (respecting break after Slot 3):
  1-slot session → start slots [1, 2, 3, 4, 5, 6] → occupies [[1], [2], [3], [4], [5], [6]]
  2-slot session → start slots [1, 2, 4, 5] → occupies [[1, 2], [2, 3], [4, 5], [5, 6]]
  3-slot session → start slots [1, 4] → occupies [[1, 2, 3], [4, 5, 6]]


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONSTRAINT CHECKING FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def slots_occupied(start_slot: int, slots_required: int) -> list[int]:
    """Return the full list of slot indices a session occupies."""
    return list(range(start_slot, start_slot + slots_required))


def check_room_conflict(assignment: dict, day: str, start_slot: int,
                        slots_required: int, room: str) -> bool:
    """
    C1 — Room Conflict check.
    Returns True if the proposed (day, slots, room) is FREE (no conflict),
    False if another already-scheduled session occupies the same room+slot.
    """
    proposed_slots = slots_occupied(start_slot, slots_required)
    for existing in assignment.values():
        if existing["Day"] == day and existing["Room"] == room:
            existing_slots = slots_occupied(existing["Start Slot"], existing["Slots Required"])
            # Check for any overlap between proposed and existing slot sets
            if set(proposed_slots) & set(existing_slots):
                return False   # CONFLICT
    return True   # FREE


def clean_name(name: str) -> str:
    """Helper function to remove titles so we just compare the core name."""
    # Removes 'Dr.', 'Dr ', and any extra spaces
    return name.replace("Dr.", "").replace("Dr ", "").strip()

def check_instructor_conflict(assignment: dict, day: str, start_slot: int,
                              slots_required: int, instructor: str) -> bool:
    """
    C6 — Instructor Conflict check.
    Handles multiple instructors split by '&' and ignores 'Dr.' prefixes.
    """
    proposed_slots = slots_occupied(start_slot, slots_required)

    # 1. Split the proposed instructors by '&' and clean the names
    proposed_names = [clean_name(n) for n in instructor.split("&")]

    for existing in assignment.values():
        if existing["Day"] == day:
            # 2. Split the existing scheduled instructors by '&' and clean them
            existing_names = [clean_name(n) for n in existing["Instructor"].split("&")]

            # 3. Check if there is ANY overlap between the two lists of names
            if set(proposed_names) & set(existing_names):
                existing_slots = slots_occupied(existing["Start Slot"], existing["Slots Required"])
                if set(proposed_slots) & set(existing_slots):
                    return False   # INSTRUCTOR BUSY (Conflict found!)

    return True   # INSTRUCTOR FREE


def is_assignment_valid(assignment: dict, session_id: str, day: str,
                        start_slot: int, slots_required: int,
                        room: str, instructor: str) -> bool:
    """
    Master constraint checker. Returns True only if ALL hard constraints pass.
    C5 is pre-filtered via VALID_STARTS so we don't recheck it here.
    """
    # C1: Room must be free
    if not check_room_conflict(assignment, day, start_slot, slots_required, room):
        return False
    # C6: Instructor must be free
    if not check_instructor_conflict(assignment, day, start_slot, slots_required, instructor):
        return False
    return True


print("✅ Constraint checker functions defined.")

✅ Constraint checker functions defined.


## Step 5 — Soft Constraint: Doctor Day-Off Scoring

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SOFT CONSTRAINT S1 — DOCTOR DAY-OFF
# We score candidate (day, slot, room) assignments by how well they
# preserve a full day off for each instructor.
#
# Strategy:
#   1. Track how many days each instructor is ALREADY scheduled on.
#   2. When choosing where to place a new session, prefer days on which the
#      instructor is ALREADY teaching (consolidating their schedule) rather
#      than spreading them across all 5 days.
#   3. This naturally leaves at least one day free per instructor.
# ─────────────────────────────────────────────────────────────────────────────

def instructor_days_used(assignment: dict, instructor: str) -> set:
    """Return the set of days on which an instructor already has sessions."""
    return {v["Day"] for v in assignment.values() if v["Instructor"] == instructor}


def day_off_score(assignment: dict, instructor: str, day: str) -> int:
    """
    S1 scoring — returns a priority score for placing a session on `day`.
    Higher score = BETTER choice (consolidates schedule, preserves a day off).

    Score logic:
      +2  if instructor is already scheduled on this day
            (reinforces consolidation — good)
      0   if instructor is not yet on this day but has fewer than 4 days used
            (acceptable — still leaves room for a day off)
      -1  if placing here would mean the instructor has NO days off
            (undesirable — soft penalty)
    """
    days_used = instructor_days_used(assignment, instructor)
    if day in days_used:
        return 2    # already teaching this day → consolidate
    elif len(days_used) < len(DAYS) - 1:   # < 4 days → still safe
        return 0
    else:
        return -1   # would eliminate the last day off


print("✅ Day-off soft constraint scorer defined.")

✅ Day-off soft constraint scorer defined.


## Step 6 — Candidate Generation & Variable Ordering

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CANDIDATE VALUE GENERATION
# For each unscheduled session we generate all (Day, Start Slot, Room)
# triples that are worth trying, ranked by:
#   1. Day-off soft score (S1) — descending (better first)
#   2. Random shuffle within equal-score groups → balanced spread
#
# This randomised ordering prevents every session piling onto Sunday.
# ─────────────────────────────────────────────────────────────────────────────

def generate_candidates(session: pd.Series, assignment: dict) -> list[tuple]:
    slots_req  = session["Slots Required"]
    instructor = session["Instructor"]
    sess_type  = session["Session Type"]

    # --- STRICT ROOM LOGIC FOR ALL TYPES ---
    # No more "fallbacks" to other room categories!
    if sess_type == "Lab":
        room_order = LABS.copy()
    elif sess_type == "Lecture":
        room_order = LECTURE_HALLS + CLASSROOMS
    elif sess_type == "Section":
        room_order = CLASSROOMS
    else:
        # Just in case there's a type in the Excel file like "Seminar"
        room_order = ALL_ROOMS
    # ---------------------------------------

    # Pre-computed valid start slots (C5 already baked in)
    valid_starts = VALID_STARTS[slots_req]

    candidates = []
    # Randomise day and room order within each pass so the schedule spreads
    days_shuffled  = DAYS[:]
    random.shuffle(days_shuffled)

    # Randomise the strictly filtered rooms
    rooms_shuffled = room_order[:]
    random.shuffle(rooms_shuffled)

    for day in days_shuffled:
        score = day_off_score(assignment, instructor, day)
        for start_slot in valid_starts:
            for room in rooms_shuffled:
                candidates.append((score, day, start_slot, room))

    # Sort: higher soft score first; ties broken by random insertion order
    candidates.sort(key=lambda x: -x[0])
    return candidates


def order_sessions(sessions: pd.DataFrame) -> list:
    """
    Minimum Remaining Values (MRV) heuristic — schedule the HARDEST sessions first.
    Hardest = sessions requiring the most consecutive slots (3-slot > 2-slot > 1-slot),
    then by number of valid start slots (fewest first).
    This reduces backtracking by committing constrained sessions early.
    """
    return sessions.sort_values(
        by="Slots Required", ascending=False
    ).index.tolist()


print("✅ Candidate generator and MRV ordering defined.")

✅ Candidate generator and MRV ordering defined.


## Step 7 — Backtracking Search Algorithm

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BACKTRACKING SEARCH
# ─────────────────────────────────────────────────────────────────────────────
# Classic recursive CSP backtracking:
#
#   backtrack(assignment, remaining_sessions)
#     if no sessions remain → return assignment  (SOLUTION FOUND)
#     pick the next session to assign (MRV order)
#     for each candidate (day, start_slot, room):
#         if all hard constraints satisfied:
#             add to assignment
#             result = backtrack(assignment, remaining_sessions - this one)
#             if result is not None → return result  (propagate success)
#             undo assignment  (BACKTRACK)
#     return None  (no valid assignment found for this branch → caller backtracks)
# ─────────────────────────────────────────────────────────────────────────────

# Global counters for diagnostics
_calls       = 0
_backtracks  = 0


def backtrack(assignment: dict, remaining_ids: list, sessions_df: pd.DataFrame) -> dict | None:
    """
    Recursive CSP Backtracking Search.

    Parameters
    ----------
    assignment     : dict mapping session_id → placement dict
    remaining_ids  : list of session row indices not yet assigned
    sessions_df    : the full sessions DataFrame (indexed by row index)

    Returns
    -------
    Complete assignment dict if a solution exists, else None.
    """
    global _calls, _backtracks
    _calls += 1

    # BASE CASE: all sessions assigned → solution found!
    if not remaining_ids:
        return assignment

    # Progress indicator every 50 recursive calls
    if _calls % 50 == 0:
        scheduled = len(sessions_df) - len(remaining_ids)
        print(f"  [Progress] Scheduled {scheduled}/{len(sessions_df)} | "
              f"Calls: {_calls} | Backtracks: {_backtracks}", end="\r")

    # VARIABLE SELECTION: take the first from MRV-ordered list
    current_idx = remaining_ids[0]
    session     = sessions_df.loc[current_idx]
    session_id  = session["Session ID"]
    instructor  = session["Instructor"]
    slots_req   = session["Slots Required"]

    # VALUE ORDERING: generate ranked candidates (LCV + soft score + randomisation)
    candidates = generate_candidates(session, assignment)

    for (_, day, start_slot, room) in candidates:

        # ── CHECK ALL HARD CONSTRAINTS ──────────────────────────────────────
        # C5 is already guaranteed by start_slot ∈ VALID_STARTS[slots_req]
        # C1 and C6 are checked inside is_assignment_valid
        if is_assignment_valid(assignment, session_id, day, start_slot,
                               slots_req, room, instructor):

            # ASSIGN — record placement
            assignment[session_id] = {
                "Session ID":     session_id,
                "Course Code":    session["Course Code"],
                "Course Name":    session["Course Name"],
                "Instructor":     instructor,
                "Session Type":   session["Session Type"],
                "Day":            day,
                "Start Slot":     start_slot,
                "Slots Required": slots_req,
                "Room":           room,
            }

            # RECURSE on remaining sessions
            result = backtrack(assignment, remaining_ids[1:], sessions_df)

            if result is not None:
                return result   # ← propagate success upward

            # BACKTRACK — undo this assignment and try next candidate
            del assignment[session_id]
            _backtracks += 1

    # No valid candidate worked → signal failure to caller
    return None


print("✅ Backtracking Search algorithm defined.")

✅ Backtracking Search algorithm defined.


## Step 8 — Run the Scheduler

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN THE CSP SCHEDULER
# ─────────────────────────────────────────────────────────────────────────────

# Reset diagnostic counters
_calls      = 0
_backtracks = 0

# Order sessions by MRV (hardest first: 3-slot > 2-slot > 1-slot)
sessions_df  = df.set_index(df.index)  # keep original int index
ordered_ids  = order_sessions(sessions_df)

print(f"🚀 Starting CSP Backtracking Search for {len(ordered_ids)} sessions…")
print(f"   Sessions by slots required:")
for k, v in sessions_df["Slots Required"].value_counts().sort_index(ascending=False).items():
    print(f"     {k}-slot sessions: {v}")
print()

start_time = time.time()

random.seed(42)   # fix seed for reproducibility; remove or change for variation
solution = backtrack({}, ordered_ids, sessions_df)

elapsed = time.time() - start_time

print()  # newline after \r progress
if solution:
    print(f"✅ SOLUTION FOUND in {elapsed:.2f}s")
    print(f"   Total recursive calls : {_calls:,}")
    print(f"   Total backtracks       : {_backtracks:,}")
    print(f"   Sessions scheduled     : {len(solution)}/{len(df)}")
else:
    print("❌ NO SOLUTION FOUND — The problem may be over-constrained.")
    print(f"   Calls: {_calls:,} | Backtracks: {_backtracks:,}")

🚀 Starting CSP Backtracking Search for 160 sessions…
   Sessions by slots required:
     3-slot sessions: 39
     2-slot sessions: 84
     1-slot sessions: 37

  [Progress] Scheduled 149/160 | Calls: 150 | Backtracks: 0
✅ SOLUTION FOUND in 0.12s
   Total recursive calls : 161
   Total backtracks       : 0
   Sessions scheduled     : 160/160


## Step 9 — Build Output DataFrame

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BUILD HUMAN-READABLE OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

def format_time_block(start_slot: int, slots_required: int) -> str:
    """
    Convert (start_slot, slots_required) → a human-readable time range string.
    Example: start_slot=1, slots_required=2 → "09:00-10:40"
    """
    end_slot = start_slot + slots_required - 1
    # Extract start time from first slot, end time from last slot
    start_time = SLOT_TIMES[start_slot].split("-")[0]
    end_time   = SLOT_TIMES[end_slot].split("-")[1]
    slot_labels = ", ".join([f"Slot {s}" for s in range(start_slot, end_slot + 1)])
    return f"{start_time}-{end_time} ({slot_labels})"


if solution:
    records = []
    for sess_id, info in solution.items():
        records.append({
            "Session ID":     info["Session ID"],
            "Course Code":    info["Course Code"],
            "Course Name":    info["Course Name"],
            "Instructor":     info["Instructor"],
            "Session Type":   info["Session Type"],
            "Day":            info["Day"],
            "Time Block":     format_time_block(info["Start Slot"], info["Slots Required"]),
            "Start Slot":     info["Start Slot"],
            "Slots Required": info["Slots Required"],
            "Room":           info["Room"],
        })

    results_df = pd.DataFrame(records)

    # Sort by Day → Start Slot → Room for readability
    day_order = {d: i for i, d in enumerate(DAYS)}
    results_df["Day Order"] = results_df["Day"].map(day_order)
    results_df = results_df.sort_values(["Day Order", "Start Slot", "Room"]).drop(columns=["Day Order"])
    results_df = results_df.reset_index(drop=True)

    print(f"✅ Output table built: {len(results_df)} rows")
    print()
    print("Sample output (first 10 rows):")
    display_cols = ["Session ID", "Course Code", "Instructor", "Session Type", "Day", "Time Block", "Room"]
    results_df[display_cols].head(10)

✅ Output table built: 160 rows

Sample output (first 10 rows):


## Step 10 — Constraint Violation Audit

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONSTRAINT VIOLATION AUDIT
# Verify the final schedule satisfies all hard constraints.
# ─────────────────────────────────────────────────────────────────────────────

def audit_schedule(results_df: pd.DataFrame):
    """Audit the schedule for any hard constraint violations and print a report."""
    violations = {"C1_Room_Conflict": [], "C5_Break_Violation": [], "C6_Instructor_Conflict": []}

    rows = results_df.to_dict("records")

    for i, r1 in enumerate(rows):
        slots1 = set(range(r1["Start Slot"], r1["Start Slot"] + r1["Slots Required"]))

        # C5: Check break crossing (slot 3 AND slot 4 in same session)
        if 3 in slots1 and 4 in slots1:
            violations["C5_Break_Violation"].append(
                f"  ❌ {r1['Session ID']} ({r1['Course Code']}) crosses break on {r1['Day']}"
            )

        for j, r2 in enumerate(rows):
            if j <= i:
                continue
            slots2 = set(range(r2["Start Slot"], r2["Start Slot"] + r2["Slots Required"]))
            overlap = slots1 & slots2

            if r1["Day"] == r2["Day"] and overlap:
                # C1: Same room, same day, overlapping slots
                if r1["Room"] == r2["Room"]:
                    violations["C1_Room_Conflict"].append(
                        f"  ❌ {r1['Session ID']} & {r2['Session ID']} → "
                        f"Room {r1['Room']} on {r1['Day']} slots {overlap}"
                    )
                # C6: Same instructor, same day, overlapping slots
                if r1["Instructor"] == r2["Instructor"]:
                    violations["C6_Instructor_Conflict"].append(
                        f"  ❌ {r1['Instructor']} → "
                        f"{r1['Session ID']} & {r2['Session ID']} on {r1['Day']} slots {overlap}"
                    )

    print("═" * 55)
    print("  HARD CONSTRAINT AUDIT REPORT")
    print("═" * 55)
    for constraint, issues in violations.items():
        status = "✅ PASS" if not issues else f"❌ FAIL ({len(issues)} violations)"
        print(f"  {constraint}: {status}")
        for msg in issues[:5]:   # show at most 5 per type
            print(msg)
    print("═" * 55)

    total = sum(len(v) for v in violations.values())
    if total == 0:
        print("  🎉 ALL HARD CONSTRAINTS SATISFIED — Schedule is valid!")
    else:
        print(f"  ⚠️  {total} violation(s) found.")
    print("═" * 55)


if solution:
    audit_schedule(results_df)

═══════════════════════════════════════════════════════
  HARD CONSTRAINT AUDIT REPORT
═══════════════════════════════════════════════════════
  C1_Room_Conflict: ✅ PASS
  C5_Break_Violation: ✅ PASS
  C6_Instructor_Conflict: ✅ PASS
═══════════════════════════════════════════════════════
  🎉 ALL HARD CONSTRAINTS SATISFIED — Schedule is valid!
═══════════════════════════════════════════════════════


## Step 11 — Soft Constraint (S1) Day-Off Report

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# S1 — INSTRUCTOR DAY-OFF REPORT
# ─────────────────────────────────────────────────────────────────────────────

if solution:
    instructor_days = results_df.groupby("Instructor")["Day"].apply(set)

    has_day_off   = []
    no_day_off    = []

    for instructor, days_used in instructor_days.items():
        days_off = set(DAYS) - days_used
        if days_off:
            has_day_off.append((instructor, sorted(days_off, key=DAYS.index)))
        else:
            no_day_off.append(instructor)

    total_instructors = len(instructor_days)
    pct = 100 * len(has_day_off) / total_instructors if total_instructors else 0

    print(f"  S1 — Doctor Day-Off Optimisation Report")
    print(f"  ─────────────────────────────────────────────")
    print(f"  Total instructors   : {total_instructors}")
    print(f"  With ≥1 day off     : {len(has_day_off)} ({pct:.1f}%)")
    print(f"  With NO day off     : {len(no_day_off)}")
    print()

    if no_day_off:
        print("  Instructors with NO day off (scheduled all 5 days):")
        for name in no_day_off:
            print(f"    • {name}")
    else:
        print("  🎉 ALL instructors have at least one day off!")

    print()
    print("  Sample — instructors with their days off:")
    for instructor, days_off in has_day_off[:10]:
        print(f"    {instructor:<30} → off on {', '.join(days_off)}")

  S1 — Doctor Day-Off Optimisation Report
  ─────────────────────────────────────────────
  Total instructors   : 66
  With ≥1 day off     : 66 (100.0%)
  With NO day off     : 0

  🎉 ALL instructors have at least one day off!

  Sample — instructors with their days off:
    Abdelkader                     → off on Sunday, Monday, Tuesday, Thursday
    Alaa Nagib                     → off on Sunday, Monday, Wednesday, Thursday
    Amr                            → off on Sunday, Monday, Wednesday, Thursday
    Dr. Amany                      → off on Monday, Wednesday, Thursday
    Dr. Ayat                       → off on Monday, Tuesday, Wednesday, Thursday
    Dr. Dina                       → off on Sunday, Monday, Wednesday, Thursday
    Dr. Ehab                       → off on Monday, Tuesday, Wednesday
    Dr. Eman                       → off on Sunday, Monday, Wednesday, Thursday
    Dr. Engy                       → off on Tuesday, Wednesday
    Dr. Farouk                     → off on

## Step 12 — Schedule Statistics & Visualisation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SCHEDULE STATISTICS
# ─────────────────────────────────────────────────────────────────────────────

if solution:
    print("═" * 55)
    print("  SCHEDULE STATISTICS")
    print("═" * 55)

    # Sessions per day
    print("\n  Sessions per day:")
    day_counts = results_df["Day"].value_counts().reindex(DAYS, fill_value=0)
    for day, count in day_counts.items():
        bar = "█" * (count // 2)
        print(f"    {day:<12} {count:>3}  {bar}")

    # Sessions per session type
    print("\n  Sessions by type:")
    type_counts = results_df["Session Type"].value_counts()
    for t, c in type_counts.items():
        print(f"    {t:<12} {c}")

    # Room utilisation
    print("\n  Room utilisation (top 10 busiest rooms):")
    room_usage = results_df["Room"].value_counts().head(10)
    for room, count in room_usage.items():
        bar = "█" * count
        print(f"    {room:<10} {count:>3}  {bar}")

    # Slot distribution
    print("\n  Start-slot distribution:")
    slot_counts = results_df["Start Slot"].value_counts().sort_index()
    for slot, count in slot_counts.items():
        bar = "█" * (count // 2)
        print(f"    Slot {slot} ({SLOT_TIMES[slot]}): {count:>3}  {bar}")

    print("═" * 55)

═══════════════════════════════════════════════════════
  SCHEDULE STATISTICS
═══════════════════════════════════════════════════════

  Sessions per day:
    Sunday        43  █████████████████████
    Monday        25  ████████████
    Tuesday       32  ████████████████
    Wednesday     36  ██████████████████
    Thursday      24  ████████████

  Sessions by type:
    Lecture      94
    Section      37
    Lab          29

  Room utilisation (top 10 busiest rooms):
    CR-2        10  ██████████
    CR-3         9  █████████
    CR-8         9  █████████
    CR-4         8  ████████
    CR-16        8  ████████
    CR-10        8  ████████
    CR-1         7  ███████
    LH-2         7  ███████
    LAB-4        7  ███████
    CR-9         7  ███████

  Start-slot distribution:
    Slot 1 (09:00-09:50):  89  ████████████████████████████████████████████
    Slot 2 (09:50-10:40):  12  ██████
    Slot 3 (10:40-11:30):   7  ███
    Slot 4 (12:10-13:00):  52  ██████████████████████████
═

## Step 13 — Export to CSV

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EXPORT TO CSV
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_FILE = "ERU_Timetable_Spring2026_CSP.csv"

if solution:
    export_cols = [
        "Session ID", "Course Code", "Course Name",
        "Instructor", "Session Type",
        "Day", "Time Block", "Room"
    ]
    results_df[export_cols].to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"✅ Timetable exported to: {OUTPUT_FILE}")
    print(f"   Rows : {len(results_df)}")
    print(f"   Columns: {export_cols}")
    print()
    print("Preview (first 15 rows):")
    results_df[export_cols].head(15)
else:
    print("❌ No solution to export.")

✅ Timetable exported to: ERU_Timetable_Spring2026_CSP.csv
   Rows : 160
   Columns: ['Session ID', 'Course Code', 'Course Name', 'Instructor', 'Session Type', 'Day', 'Time Block', 'Room']

Preview (first 15 rows):


## Summary

| Component | Detail |
|---|---|
| **Algorithm** | CSP Backtracking Search |
| **Variable ordering** | MRV — most-constrained sessions first (3-slot → 2-slot → 1-slot) |
| **Value ordering** | LCV + S1 day-off score + random shuffle for balance |
| **Hard constraints** | C1 Room Conflict, C5 Break Enforcement, C6 Instructor Conflict |
| **Soft constraint** | S1 Doctor Day-Off (scored, not hard-enforced) |
| **Output** | `ERU_Timetable_Spring2026_CSP.csv` |